# In Class Assignment 8


Ethan Ho  
CMPE 256  
Professor Eirinaki  
March 2, 2026  

In [ ]:
from surprise import Dataset
from surprise import Reader
from surprise.model_selection.split import train_test_split
from surprise.model_selection import GridSearchCV
import pandas as pd
from surprise import KNNBasic, KNNWithMeans
from surprise import SVD

from collections import defaultdict

In [3]:
ratings_df  = pd.read_csv('datasets/movie_night_utilitymatrix_S26.csv')
reader = Reader(rating_scale=(1,5))
data=Dataset.load_from_df(ratings_df,reader)

In [4]:
trainingSet, testSet = train_test_split(data, test_size=0.3, train_size=None, random_state=None, shuffle=True)

In [5]:
param_grid = {'k': [3, 5, 10, 20],
              'sim_options': {'name': ['pearson', 'cosine', 'msd', 'pearson_baseline'],
                              'min_support': [1, 2, 3, 5],
                              'user_based': [False, True],
                              'shrinkage': [25, 50, 75, 100]} # only applies to pearson baseline
              }

In [ ]:
basic_KNN_gs = GridSearchCV(KNNBasic, param_grid, measures=['rmse'], cv=5)

In [7]:
basic_KNN_gs.fit(data)

Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.

In [8]:
print(basic_KNN_gs.best_score['rmse'])
print(basic_KNN_gs.best_params['rmse'])

1.00772154278823
{'k': 20, 'sim_options': {'name': 'pearson_baseline', 'min_support': 1, 'user_based': True, 'shrinkage': 50}}


In [10]:
means_KNN_gs = GridSearchCV(KNNWithMeans, param_grid, measures=['rmse'], cv=5)
means_KNN_gs.fit(data)

Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.

In [11]:
print(means_KNN_gs.best_score['rmse'])
print(means_KNN_gs.best_params['rmse'])

0.9949747289178219
{'k': 20, 'sim_options': {'name': 'pearson_baseline', 'min_support': 1, 'user_based': True, 'shrinkage': 25}}


In [15]:
mf_param_grid = {'n_factors': [5, 10, 20, 50, 100],
              'reg_all': [0.001, 0.005, 0.01, 0.1],
              'n_epochs': [10, 20, 50, 100]
              }

In [16]:
mf_gs = GridSearchCV(SVD, mf_param_grid, measures=['rmse'], cv=5) 
mf_gs.fit(data)

In [17]:
print(mf_gs.best_score['rmse'])
print(mf_gs.best_params['rmse'])

0.9786717039721516
{'n_factors': 50, 'reg_all': 0.1, 'n_epochs': 100}


## Model Evaluations

The three top performing models and their paramaters are:   

##### Basic KNN

RMSE: 1.00772154278823   
Parameters: {'k': 20, 'sim_options': {'name': 'pearson_baseline', 'min_support': 1, 'user_based': True, 'shrinkage': 50}}  

##### KNN with Means

RMSE: 0.9949747289178219    
Parameters: {'k': 20, 'sim_options': {'name': 'pearson_baseline', 'min_support': 1, 'user_based': True, 'shrinkage': 25}}  

##### MF

RMSE: 0.9786717039721516  
Parameters: {'n_factors': 50, 'reg_all': 0.1, 'n_epochs': 100}  

##### Best Model

The best performance based off RMSE would go to the SVD Matrix Factorization method which we will continue to use for our rating predictions

In [18]:
svd = mf_gs.best_estimator['rmse']
svd.fit(data.build_full_trainset())

## Generating Recommendations

In [19]:
user_df = pd.read_csv("datasets/user_name.csv")
user_dict = {}
for i in range(len(user_df)):
    user_dict[user_df.iloc[i].username] = user_df.iloc[i].id

In [20]:
movie_df = pd.read_csv("datasets/movie_name.csv")
movie_dict = {}
for i in range(len(movie_df)):
    movie_dict[movie_df.iloc[i].id] = movie_df.iloc[i].movieName

In [21]:
trainset = data.build_full_trainset()
anti_test_set = trainset.build_anti_testset()

In [22]:
predictions = svd.test(anti_test_set)

In [24]:
def getMovieRecommendations(topN):
    top_recs = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions: 
        top_recs[uid].append((iid, est))
     
    for uid, user_ratings in top_recs.items():
        user_ratings.sort(key = lambda x: x[1], reverse = True)
        top_recs[uid] = user_ratings[:topN]
     
    return top_recs

In [25]:
recommendations = getMovieRecommendations(3)

In [26]:
def getMovieName(movie_id):
    if movie_id not in movie_dict:
        return ""
    m = movie_dict[movie_id].split('[')
    temp = m[1].split(']')
    return temp[0]

In [27]:
def getMovieRecommendationsForUser(userId, recommendations):
    if userId not in user_dict:
        print("User id is not present")
        return
    u_id = user_dict[userId]
    recommended_movies = recommendations[u_id]
    movie_list = []
    for movie in recommended_movies:
        movie_list.append((getMovieName(movie[0]),movie[1]))
    return movie_list    

In [29]:
getMovieRecommendationsForUser('SPRING-26-582',recommendations)

[('The Lord of the Rings', 4.508131942141253),
 ('Fight Club', 4.215220262902759),
 ('Captain America', 4.068143551514575)]